# NDT7 (M-Lab) Data Prep — Cambodia Broadband + Mobile, Province x Quarter

Aggregates `../../../data/ndt7/kh/mlab_kh_clean.parquet` (770,501 raw NDT7 test records, already ISP-classified and province-joined via
per-IP lookup + point-in-polygon) into province x quarter format, split into Broadband and
Mobile/Cellular parts, mirroring the same structure across all three NDT7 "tigger" countries
(Cambodia/Thailand/Vietnam).

**Rebuilt to use DuckDB instead of a manual pyarrow-batch-streaming loop** — DuckDB reads the
parquet file directly and does the tile-binning + GROUP BY aggregation out-of-core (no manual
batching code needed, no risk of the memory issues the streaming version was written to avoid).
The tile-binning and weighted-aggregation formulas are byte-for-byte unchanged from the pandas
version — verified against the prior pandas-based export (float-precision-only differences,
~1e-13, from AVG() accumulation order).

**Province name mapping still applied** — raw parquet 'province' values use Khmer-romanized diacritics that don't match `cambodia_reference.csv`; the mapping is applied in pandas *after* DuckDB's tile-level aggregation (on the much smaller intermediate result), not pushed into SQL — same 25-entry `PROVINCE_MAP` dict as before.

Same tile scheme as Ookla's own published tiles (zoom-16 slippy tiles, ~610m) — keeps
`n_tiles`/`is_reliable` comparable across Ookla and NDT7, and across countries:
`total_tests >= 100 & n_tiles >= 5`.

**Outputs:**
- `data/exports/ndt7_cambodia_province_quarterly.csv` — Broadband
- `data/exports/ndt7_mobile_cambodia_province_quarterly.csv` — Mobile/Cellular
  (renamed from `ndt7_cambodia_mobile_...` to match Ookla's `ookla_mobile_<country>_...`
  naming convention — position of "mobile" now matches across both pipelines)

In [1]:
import duckdb
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../../data/ndt7/kh/mlab_kh_clean.parquet'
KH_REF_CSV = '../../../data/reference/cambodia_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3

### 1. Tile-Binning + Province-Quarter Aggregation (DuckDB)

All heavy row-level work (filtering, quarter-labeling, zoom-16 mercator tile assignment, GROUP BY tile x quarter x type x network_type) happens in one DuckDB SQL query against the raw parquet — no Python-side batching.

In [2]:
sql = f"""
WITH filtered AS (
    SELECT
        mean_throughput_mbps,
        LEAST(min_rtt, 2000) AS min_rtt,
        latitude, longitude, type, network_type, province,
        date_part('year', date) AS yr,
        date_part('quarter', date) AS qtr
    FROM read_parquet('{RAW_PARQUET}')
    WHERE mean_throughput_mbps > 0
      AND latitude IS NOT NULL AND longitude IS NOT NULL
      AND province IS NOT NULL AND date IS NOT NULL
),
tiled AS (
    SELECT
        *,
        (CAST(yr AS VARCHAR) || '-Q' || CAST(qtr AS VARCHAR)) AS year_q,
        CAST(FLOOR((longitude + 180) / 360 * {N_TILES}) AS BIGINT) AS tile_x_raw,
        CAST(FLOOR((1 - (ln(tan(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878))) + 1.0/cos(radians(LEAST(GREATEST(latitude, -85.05112878), 85.05112878)))) ) / pi()) / 2 * {N_TILES}) AS BIGINT) AS tile_y_raw
    FROM filtered
),
clipped AS (
    SELECT *,
        LEAST(GREATEST(tile_x_raw, 0), {N_TILES}-1) AS tile_x,
        LEAST(GREATEST(tile_y_raw, 0), {N_TILES}-1) AS tile_y
    FROM tiled
),
tile_id_cte AS (
    SELECT *, (CAST(tile_x AS VARCHAR) || '_' || CAST(tile_y AS VARCHAR)) AS tile_id
    FROM clipped
),
tile_agg AS (
    SELECT
        year_q, tile_id, type, network_type,
        AVG(mean_throughput_mbps) AS tile_mean,
        AVG(min_rtt) AS tile_lat,
        COUNT(*) AS test_count,
        mode(province) AS province
    FROM tile_id_cte
    GROUP BY year_q, tile_id, type, network_type
    HAVING COUNT(*) >= {MIN_TILE_TESTS}
)
SELECT * FROM tile_agg
"""

con = duckdb.connect()
tile_agg_all = con.execute(sql).df()
print(f"Tile x quarter x type x network rows (>= {MIN_TILE_TESTS} tests/tile): {len(tile_agg_all):,}")
print(f"Quarters covered: {sorted(tile_agg_all['year_q'].unique())}")
print(tile_agg_all['network_type'].value_counts())

Tile x quarter x type x network rows (>= 3 tests/tile): 494
Quarters covered: ['2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4', '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4', '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4']
network_type
broadband    313
cellular      93
hosting       88
Name: count, dtype: int64


### 2. Province Name Mapping — Raw (Khmer-romanized) → Reference (`province_en`)

Applied here, after DuckDB's tile-level aggregation — the intermediate result is small (thousands of rows, not hundreds of thousands), so this stays a plain pandas `.map()` exactly like the original.

In [3]:
# Raw parquet 'province' values (diacritic Khmer romanization) -> cambodia_reference.csv 'province_en'
PROVINCE_MAP = {
    'Batdâmbâng': 'Battambang',
    'BântéayMéanchey': 'Bantey Meanchey',
    'KaôhKong': 'Koh Kong',
    'Kep': 'Kep',
    'KrongPailin': 'Pailin',
    'KrongPreahSihanouk': 'Preah Sihanouk',
    'Krâchéh': 'Kratie',
    'KâmpóngCham': 'Kampong Cham',
    'KâmpóngChhnang': 'Kampong Chhnang',
    'KâmpóngSpœ': 'Kampong Speu',
    'KâmpóngThum': 'Kampong Thom',
    'Kâmpôt': 'Kampot',
    'Kândal': 'Kandal',
    'MôndólKiri': 'Mondulkiri',
    'OtdarMeanChey': 'Oddar Meanchey',
    'PhnomPenh': 'Phnom Penh',
    'Pouthisat': 'Pursat',
    'PreahVihéar': 'Preah Vihear',
    'PreyVêng': 'Prey Veng',
    'Rôtânôkiri': 'Ratanakiri Province',
    'Siemréab': 'Siem Reap',
    'StœngTrêng': 'Stung Treng',
    'SvayRieng': 'Svay Rieng',
    'Takêv': 'Takeo',
    'TbongKhmum': 'Tbong Khmum',
}

ref_check = pd.read_csv(KH_REF_CSV)
unmapped_targets = set(PROVINCE_MAP.values()) - set(ref_check['province_en'])
print(f"Mapping covers {len(PROVINCE_MAP)} raw province names -> {len(set(PROVINCE_MAP.values()))} reference provinces")
if unmapped_targets:
    print(f"WARNING — mapped targets not found in reference: {unmapped_targets}")
else:
    print("All mapped targets found in reference.")

tile_agg_all['province'] = tile_agg_all['province'].map(PROVINCE_MAP)
tile_agg_all = tile_agg_all.dropna(subset=['province'])
print(f"Rows after province mapping: {len(tile_agg_all):,}")

Mapping covers 25 raw province names -> 25 reference provinces
All mapped targets found in reference.
Rows after province mapping: 494


### 3. Province-Level Weighted Aggregation (per network type)

In [4]:
def build_province_quarterly(tile_agg_all, network_type, ref):
    tile_agg = tile_agg_all[tile_agg_all['network_type'] == network_type]
    print(f"[{network_type}] tile x quarter x type rows: {len(tile_agg):,}")

    dl = tile_agg[tile_agg['type'] == 'download']
    ul = tile_agg[tile_agg['type'] == 'upload']

    dl_stats = dl.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_d_mbps': np.average(g['tile_mean'], weights=g['test_count']),
        'avg_lat_ms_wt': np.average(g['tile_lat'], weights=g['test_count']),
        'total_tests': g['test_count'].sum(),
        'n_tiles': g['tile_id'].nunique(),
    }), include_groups=False).reset_index()

    ul_stats = ul.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_u_mbps': np.average(g['tile_mean'], weights=g['test_count']),
    }), include_groups=False).reset_index()

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    master['is_reliable'] = (master['total_tests'] >= 100) & (master['n_tiles'] >= 5)
    print(f"[{network_type}] province x quarter rows: {len(master)} | reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — provinces with no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

In [5]:
ref = pd.read_csv(KH_REF_CSV)

---
## Part 1 — Broadband

In [6]:
broadband_master = build_province_quarterly(tile_agg_all, 'broadband', ref)
broadband_master.head()

[broadband] tile x quarter x type rows: 313
[broadband] province x quarter rows: 151 | reliable: 0 (0.0%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Bantey Meanchey,5.910550,85.844847,59.0,1.0,11.251138,2023,1,False,Northwest,1,898484,2167.4,135,6225.53,199216.96
1,2023-Q1,Battambang,29.017656,84.042333,6.0,1.0,12.997123,2023,1,False,Northwest,2,1132017,2167.4,97,6225.53,199216.96
2,2023-Q1,Kampong Cham,14.432567,219.129500,8.0,1.0,4.936569,2023,1,False,East,4,1062914,2167.4,234,6225.53,199216.96
3,2023-Q1,Kampong Speu,7.574400,75.383429,14.0,1.0,0.900915,2023,1,False,Southwest,4,924175,2167.4,132,6225.53,199216.96
4,2023-Q1,Kampong Thom,19.127098,104.520349,2645.0,1.0,12.449575,2023,1,False,Center,3,807254,2167.4,58,6225.53,199216.96


In [7]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../../data/exports/ndt7_cambodia_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 151 rows -> ../../../data/exports/ndt7_cambodia_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Bantey Meanchey,2023-Q1,2023,1,5.910550,11.251138,85.844847,59.0,1.0,False,Northwest,1,898484,2167.4,135,6225.53,199216.96
1,Battambang,2023-Q1,2023,1,29.017656,12.997123,84.042333,6.0,1.0,False,Northwest,2,1132017,2167.4,97,6225.53,199216.96
2,Kampong Cham,2023-Q1,2023,1,14.432567,4.936569,219.129500,8.0,1.0,False,East,4,1062914,2167.4,234,6225.53,199216.96


---
## Part 2 — Mobile/Cellular

In [8]:
mobile_master = build_province_quarterly(tile_agg_all, 'cellular', ref)
mobile_master.head()

[cellular] tile x quarter x type rows: 93


[cellular] province x quarter rows: 43 | reliable: 0 (0.0%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Kampong Thom,16.942150,123.476180,13594.0,1.0,5.481035,2023,1,False,Center,3,807254,2167.4,58,6225.53,199216.96
1,2023-Q1,Phnom Penh,14.541996,169.373942,2192.0,1.0,6.955191,2023,1,False,Capital,1,2352851,2167.4,3465,6225.53,199216.96
2,2023-Q1,Pursat,12.099882,85.531800,5.0,1.0,6.052613,2023,1,False,West,4,516072,2167.4,41,6225.53,199216.96
3,2023-Q1,Takeo,15.466607,106.679075,40.0,1.0,8.219753,2023,1,False,South,1,1097243,2167.4,308,6225.53,199216.96
4,2023-Q2,Kampong Thom,11.680324,105.550784,31137.0,1.0,7.630777,2023,2,False,Center,3,807254,2167.4,58,6225.53,199216.96


In [9]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../../data/exports/ndt7_mobile_cambodia_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 43 rows -> ../../../data/exports/ndt7_mobile_cambodia_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Kampong Thom,2023-Q1,2023,1,16.942150,5.481035,123.476180,13594.0,1.0,False,Center,3,807254,2167.4,58,6225.53,199216.96
1,Phnom Penh,2023-Q1,2023,1,14.541996,6.955191,169.373942,2192.0,1.0,False,Capital,1,2352851,2167.4,3465,6225.53,199216.96
2,Pursat,2023-Q1,2023,1,12.099882,6.052613,85.531800,5.0,1.0,False,West,4,516072,2167.4,41,6225.53,199216.96


## Summary

- Input: Cambodia NDT7 raw test records, already province-joined + ISP-classified
- Output: province x quarter aggregates for Broadband and Mobile separately, tile-binned at
  Ookla's zoom-16 resolution, same `is_reliable` threshold as every Ookla country notebook and
  the other NDT7 "tigger" prep notebooks
- Engine: DuckDB (was: manual pyarrow-batch-streaming loop in pandas) — verified to reproduce
  the prior pandas-based export exactly (float-precision-only differences)